# Step 2: Create Delta Tables

Reads parquet files from the volume (generated in Step 1) and creates
managed Delta tables with comments.

In [ ]:
dbutils.widgets.text("catalog", "yd_launchpad_final_classic_catalog")
dbutils.widgets.text("schema", "genie_app")
dbutils.widgets.text("company_name", "NovaTech Logistics")

import re
def slugify(name: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', name.lower()).strip('_')
    if slug and slug[0].isdigit():
        slug = f'co_{slug}'
    return slug[:50]

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
COMPANY_SLUG = slugify(dbutils.widgets.get("company_name"))
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{COMPANY_SLUG}"

print(f"Creating tables in {CATALOG}.{SCHEMA} (prefix: {COMPANY_SLUG}_)")

In [ ]:
import json

# Load pipeline state from Step 1
state_raw = dbutils.fs.head(f"{VOLUME_PATH}/pipeline_state.json", 100000)
state = json.loads(state_raw)

tables = state["tables"]
COMPANY_SLUG = state["company_slug"]
print(f"Found {len(tables)} tables to create (prefix: {COMPANY_SLUG}_):")
for t in tables:
    print(f"  - {COMPANY_SLUG}_{t['name']} ({t.get('row_count', 0)} rows, {len(t['columns'])} columns)")

In [ ]:
tables_info = []

for table_def in tables:
    table_name = table_def["name"]
    prefixed_name = f"{COMPANY_SLUG}_{table_name}"
    comment = table_def.get("comment", "")
    columns = table_def["columns"]
    full_name = f"`{CATALOG}`.`{SCHEMA}`.`{prefixed_name}`"
    parquet_path = f"{VOLUME_PATH}/{table_name}/"
    
    print(f"\nCreating {full_name}...")
    
    # Drop and recreate
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    
    # Read parquet and save as managed table
    df = spark.read.parquet(parquet_path)
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{prefixed_name}")
    
    # Add table comment
    if comment:
        safe_comment = comment.replace("'", "''")
        spark.sql(f"COMMENT ON TABLE {full_name} IS '{safe_comment}'")
    
    # Add column comments
    for col in columns:
        col_comment = col.get("comment", "")
        if col_comment:
            safe_col_comment = col_comment.replace("'", "''")
            spark.sql(f"ALTER TABLE {full_name} ALTER COLUMN `{col['name']}` COMMENT '{safe_col_comment}'")
    
    row_count = spark.table(f"{CATALOG}.{SCHEMA}.{prefixed_name}").count()
    print(f"  {full_name}: {row_count:,} rows")
    
    tables_info.append({
        "full_name": f"{CATALOG}.{SCHEMA}.{prefixed_name}",
        "table_name": prefixed_name,
        "comment": comment,
    })

print(f"\nAll {len(tables_info)} tables created!")

In [ ]:
# Update pipeline state with tables_info for Step 3
state["tables_info"] = tables_info
dbutils.fs.put(f"{VOLUME_PATH}/pipeline_state.json", json.dumps(state, indent=2), overwrite=True)
print(f"Pipeline state updated with table info")